Цель ноутбука — провести расширенные NLP-эксперименты для задачи классификации мошеннических вакансий.

В этом ноутбуке:
- формируется единый текстовый признак из текстовых колонок;
- строятся TF-IDF признаки;
- обучаются несколько моделей классификации;
- проводится подбор гиперпараметров;
- сравниваются модели по Precision, Recall, F1-score и ROC-AUC;
- выбирается финальная модель для дальнейшего использования.

In [14]:
import os
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_PATH = "../data/processed/processed_jobs.csv"

In [15]:
df = pd.read_csv(DATA_PATH)

df.shape, df.head()

((17880, 18),
                                        title            location salary_range  \
 0                           Marketing Intern    US, NY, New York          NaN   
 1  Customer Service - Cloud Video Production      NZ, , Auckland          NaN   
 2    Commissioning Machinery Assistant (CMA)       US, IA, Wever          NaN   
 3          Account Executive - Washington DC  US, DC, Washington          NaN   
 4                        Bill Review Manager  US, FL, Fort Worth          NaN   
 
                                      company_profile  \
 0  <h3>We're Food52, and we've created a groundbr...   
 1  <h3>90 Seconds, the worlds Cloud Video Product...   
 2  <h3></h3>\r\n<p>Valor Services provides Workfo...   
 3  <p>Our passion for improving quality of life t...   
 4  <p>SpotSource Solutions LLC is a Global Human ...   
 
                                          description  \
 0  <p>Food52, a fast-growing, James Beard Award-w...   
 1  <p>Organised - Focused - Vibra

В датасете несколько текстовых колонок: `title`, `company_profile`, `description`, `requirements`, `benefits`.  
Для NLP-моделей объединим их в один общий текстовый признак `text`.

Также выполним базовую очистку текста:
- удалим HTML-теги;
- удалим ссылки;
- оставим только буквы, цифры и пробелы;
- приведём текст к нижнему регистру;
- удалим лишние пробелы.

In [16]:
TEXT_COLS = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits",
]

def clean_html_text(text):
    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower().strip()

for col in TEXT_COLS:
    df[col] = df[col].fillna("")

df["text"] = df[TEXT_COLS].agg(" ".join, axis=1)
df["text"] = df["text"].apply(clean_html_text)

df[["text", "fraudulent"]].head()

,text,fraudulent
0,marketing intern we re food52 and we ve create...,0
1,customer service cloud video production 90 sec...,0
2,commissioning machinery assistant cma valor se...,0
3,account executive washington dc our passion fo...,0
4,bill review manager spotsource solutions llc i...,0


Проверка баланса классов

In [17]:
df["fraudulent"].value_counts(normalize=True)

0    0.951566
1    0.048434
Name: fraudulent, dtype: float64

Разделим данные на обучающую и тестовую выборки в пропорции 80/20.

Так как классы несбалансированы, используем `stratify=y`, чтобы сохранить одинаковую долю мошеннических вакансий в train и test.  
Также фиксируем `random_state=42` для воспроизводимости экспериментов.

In [18]:
X = df["text"]
y = df["fraudulent"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_test.shape, y_train.mean(), y_test.mean()

((14304,), (3576,), 0.04844798657718121, 0.04837807606263982)

На этом этапе сравним несколько классических моделей для текстовой классификации.  
Во всех экспериментах используется один и тот же подход к признакам: `TF-IDF` по объединённому текстовому полю `text`.

Сравним следующие модели:
- Logistic Regression;
- Linear SVM;
- Multinomial Naive Bayes;
- Complement Naive Bayes;
- SGDClassifier.

In [19]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [20]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_score)
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)
        roc_auc = roc_auc_score(y_test, y_score)
    else:
        roc_auc = np.nan
    
    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_fraud": precision_score(y_test, y_pred, pos_label=1),
        "recall_fraud": recall_score(y_test, y_pred, pos_label=1),
        "f1_fraud": f1_score(y_test, y_pred, pos_label=1),
        "roc_auc": roc_auc,
    }

In [21]:
models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
    
    "Linear SVC": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)),
        ("clf", LinearSVC(
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
    
    "Multinomial NB": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)),
        ("clf", MultinomialNB()),
    ]),
    
    "Complement NB": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)),
        ("clf", ComplementNB()),
    ]),
    
    "SGDClassifier": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)),
        ("clf", SGDClassifier(
            loss="log_loss",
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
}

In [22]:
results = []

for name, model in models.items():
    print(f"Training {name}...")
    result = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(result)

results_df = pd.DataFrame(results).sort_values(by="f1_fraud", ascending=False)
results_df

Training Logistic Regression...
Training Linear SVC...
Training Multinomial NB...
Training Complement NB...
Training SGDClassifier...


,model,accuracy,precision_fraud,recall_fraud,f1_fraud,roc_auc
1,Linear SVC,0.990492,0.931677,0.867052,0.898204,0.991913
0,Logistic Regression,0.982662,0.768116,0.919075,0.836842,0.990486
4,SGDClassifier,0.977908,0.707965,0.924855,0.802005,0.989995
3,Complement NB,0.964206,0.680000,0.491329,0.570470,0.943790
2,Multinomial NB,0.962248,0.827586,0.277457,0.415584,0.943790


Вывод по базовому сравнению моделей

Лучший результат по `F1-score` для fraud-класса показала модель `Linear SVC`: `0.898`.

По сравнению с Logistic Regression, модель Linear SVC даёт более высокий precision: `0.932` против `0.768`, при этом recall остаётся достаточно высоким: `0.867`.  
Это важно для задачи fraud detection, так как модель не только находит большую часть мошеннических вакансий, но и делает меньше ложных срабатываний.

Naive Bayes модели показали худшие результаты. Особенно заметно низкое значение recall у `Multinomial NB`, что означает, что модель пропускает много мошеннических вакансий.

На следующем этапе выполним подбор гиперпараметров для лучших моделей.

Подбор гиперпараметров

После базового сравнения моделей выберем две наиболее перспективные модели для подбора гиперпараметров:

- `Linear SVC`, так как она показала лучший `F1-score`;
- `Logistic Regression`, так как она дала самый высокий recall fraud-класса.

Будем подбирать параметры TF-IDF-векторизации и регуляризации модели.  
Основная метрика оптимизации — `f1`, так как она балансирует precision и recall для fraud-класса.

In [23]:
from sklearn.model_selection import GridSearchCV

In [24]:
svc_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC(
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

svc_param_grid = {
    "tfidf__max_features": [20000, 30000, 50000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "clf__C": [0.5, 1.0, 2.0],
}

svc_grid = GridSearchCV(
    svc_pipeline,
    svc_param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

svc_grid.fit(X_train, y_train)

print("Best params:", svc_grid.best_params_)
print("Best CV F1:", svc_grid.best_score_)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best params: {'clf__C': 1.0, 'tfidf__max_features': 50000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2)}
Best CV F1: 0.8368695583390428


In [25]:
best_svc = svc_grid.best_estimator_

svc_tuned_result = evaluate_model(
    "Linear SVC tuned",
    best_svc,
    X_train,
    X_test,
    y_train,
    y_test,
)

svc_tuned_result

{'model': 'Linear SVC tuned',
 'accuracy': 0.9927293064876958,
 'precision_fraud': 0.950920245398773,
 'recall_fraud': 0.8959537572254336,
 'f1_fraud': 0.9226190476190476,
 'roc_auc': 0.9919707024913413}

Вывод по tuning Linear SVC

После подбора гиперпараметров качество Linear SVC улучшилось.

Лучшая конфигурация:
- `C = 1.0`;
- `max_features = 50000`;
- `min_df = 1`;
- `ngram_range = (1, 2)`.

На тестовой выборке tuned Linear SVC показала лучший результат среди всех моделей:
- `F1 fraud = 0.923`;
- `Precision fraud = 0.951`;
- `Recall fraud = 0.896`;
- `ROC-AUC = 0.992`.

Эта модель лучше базовой Linear SVC по F1-score и одновременно сохраняет высокий recall, что важно для задачи обнаружения мошеннических вакансий.

Подбор гиперпараметров Logistic Regression

Logistic Regression показала высокий recall fraud-класса, поэтому дополнительно подберём для неё гиперпараметры.

Цель — проверить, сможет ли модель сохранить высокий recall и улучшить F1-score.

In [26]:
logreg_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

logreg_param_grid = {
    "tfidf__max_features": [30000, 50000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "clf__C": [0.5, 1.0, 2.0, 5.0],
}

logreg_grid = GridSearchCV(
    logreg_pipeline,
    logreg_param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

logreg_grid.fit(X_train, y_train)

print("Best params:", logreg_grid.best_params_)
print("Best CV F1:", logreg_grid.best_score_)

Fitting 3 folds for each of 32 candidates, totalling 96 fits
Best params: {'clf__C': 5.0, 'tfidf__max_features': 50000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}
Best CV F1: 0.8153353022875005


In [27]:
best_logreg = logreg_grid.best_estimator_

logreg_tuned_result = evaluate_model(
    "Logistic Regression tuned",
    best_logreg,
    X_train,
    X_test,
    y_train,
    y_test,
)

logreg_tuned_result

{'model': 'Logistic Regression tuned',
 'accuracy': 0.9904921700223713,
 'precision_fraud': 0.8926553672316384,
 'recall_fraud': 0.9132947976878613,
 'f1_fraud': 0.9028571428571429,
 'roc_auc': 0.9921354670054814}

In [28]:
final_results = pd.concat([
    results_df,
    pd.DataFrame([svc_tuned_result, logreg_tuned_result])
], ignore_index=True)

final_results = final_results.sort_values(by="f1_fraud", ascending=False)
final_results

,model,accuracy,precision_fraud,recall_fraud,f1_fraud,roc_auc
5,Linear SVC tuned,0.992729,0.950920,0.895954,0.922619,0.991971
6,Logistic Regression tuned,0.990492,0.892655,0.913295,0.902857,0.992135
0,Linear SVC,0.990492,0.931677,0.867052,0.898204,0.991913
1,Logistic Regression,0.982662,0.768116,0.919075,0.836842,0.990486
2,SGDClassifier,0.977908,0.707965,0.924855,0.802005,0.989995
3,Complement NB,0.964206,0.680000,0.491329,0.570470,0.943790
4,Multinomial NB,0.962248,0.827586,0.277457,0.415584,0.943790


Финальное сравнение моделей

В итоговую таблицу вошли как базовые модели, так и модели после подбора гиперпараметров.

Лучший результат по основной метрике `F1-score` для fraud-класса показала модель `Linear SVC tuned`:

- `accuracy = 0.993`;
- `precision_fraud = 0.951`;
- `recall_fraud = 0.896`;
- `f1_fraud = 0.923`;
- `ROC-AUC = 0.992`.

По сравнению с базовой Linear SVC, подбор гиперпараметров улучшил качество:
- `F1 fraud`: с `0.898` до `0.923`;
- `Precision fraud`: с `0.932` до `0.951`;
- `Recall fraud`: с `0.867` до `0.896`.

`Logistic Regression tuned` также заметно улучшилась по сравнению с базовой версией:
- `F1 fraud`: с `0.837` до `0.903`;
- `Precision fraud`: с `0.768` до `0.893`;
- `Recall fraud`: остался высоким — `0.913`.

Несмотря на то что `Logistic Regression tuned` имеет немного более высокий recall, финальной моделью выбирается `Linear SVC tuned`, так как она показывает лучший баланс precision и recall и имеет максимальный `F1-score` для fraud-класса.

Для задачи обнаружения мошеннических вакансий это важно: модель должна находить как можно больше fraud-вакансий, но при этом не помечать слишком много реальных вакансий как мошеннические.

Детальная оценка финальной модели

На этом этапе подробнее посмотрим качество выбранной финальной модели `Linear SVC tuned`: classification report и confusion matrix.

In [29]:
final_model = best_svc

y_pred_final = final_model.predict(X_test)

print(classification_report(y_test, y_pred_final, target_names=["real", "fraud"]))

confusion_matrix(y_test, y_pred_final)

              precision    recall  f1-score   support

        real       0.99      1.00      1.00      3403
       fraud       0.95      0.90      0.92       173

    accuracy                           0.99      3576
   macro avg       0.97      0.95      0.96      3576
weighted avg       0.99      0.99      0.99      3576



array([[3395,    8],
       [  18,  155]], dtype=int64)

Интерпретация финальной модели

Финальная модель `Linear SVC tuned` показала высокое качество на тестовой выборке.

Из `173` мошеннических вакансий модель правильно нашла `155`, а `18` пропустила.  
Также модель ошибочно пометила как мошеннические только `8` реальных вакансий из `3403`.

Это хороший результат для задачи fraud detection:
- модель имеет высокий `precision` для fraud-класса — `0.95`;
- модель имеет высокий `recall` для fraud-класса — `0.90`;
- итоговый `F1-score` для fraud-класса — `0.92`.

Таким образом, модель хорошо балансирует между поиском мошеннических вакансий и контролем ложных срабатываний.

Сохранение результатов и финальной модели

Для воспроизводимости экспериментов сохраним итоговую таблицу метрик в `reports/experiments.csv`.

Также сохраним финальную модель `Linear SVC tuned` в папку `models`, чтобы использовать её на следующих этапах проекта: в отчёте, API и деплое.

In [31]:
import joblib
from pathlib import Path

REPORTS_DIR = Path("../report")
MODELS_DIR = Path("../models")

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

final_results.to_csv(REPORTS_DIR / "experiments.csv", index=False)

joblib.dump(final_model, MODELS_DIR / "final_tfidf_linear_svc.joblib")

print("Saved experiments to:", REPORTS_DIR / "experiments.csv")
print("Saved final model to:", MODELS_DIR / "final_tfidf_linear_svc.joblib")

Saved experiments to: ..\report\experiments.csv
Saved final model to: ..\models\final_tfidf_linear_svc.joblib


Проверка загрузки сохранённой модели

Проверим, что сохранённую модель можно загрузить из файла и использовать для предсказаний.  
Это важно для дальнейшего этапа CP3, где модель будет использоваться в API/деплое.

In [32]:
loaded_model = joblib.load(MODELS_DIR / "final_tfidf_linear_svc.joblib")

sample_texts = [
    "Earn money fast from home no experience required apply now",
    "We are looking for a software engineer with Python and machine learning experience",
]

loaded_predictions = loaded_model.predict(sample_texts)

for text, pred in zip(sample_texts, loaded_predictions):
    label = "fraud" if pred == 1 else "real"
    print(f"Text: {text}")
    print(f"Prediction: {label}")
    print()

Text: Earn money fast from home no experience required apply now
Prediction: fraud

Text: We are looking for a software engineer with Python and machine learning experience
Prediction: real



Сохранённая модель была успешно загружена из файла и использована для предсказаний на новых текстовых примерах.

На тестовом примере с подозрительным текстом модель предсказала `fraud`, а на обычном описании вакансии — `real`.

Это подтверждает, что модель можно использовать вне ноутбука, например в API или приложении на этапе CP3.

Анализ ошибок финальной модели

Рассмотрим ошибки финальной модели `Linear SVC tuned`.

Нас интересуют два типа ошибок:
- `false positives` — реальные вакансии, которые модель ошибочно посчитала мошенническими;
- `false negatives` — мошеннические вакансии, которые модель пропустила.

Для задачи fraud detection особенно важны false negatives, так как это мошеннические вакансии, которые остаются незамеченными.

In [33]:
errors_df = pd.DataFrame({
    "text": X_test,
    "true_label": y_test,
    "pred_label": y_pred_final,
})

false_positives = errors_df[
    (errors_df["true_label"] == 0) & (errors_df["pred_label"] == 1)
]

false_negatives = errors_df[
    (errors_df["true_label"] == 1) & (errors_df["pred_label"] == 0)
]

print("False positives:", false_positives.shape[0])
print("False negatives:", false_negatives.shape[0])

False positives: 8
False negatives: 18


False positives — это реальные вакансии, которые модель ошибочно отнесла к мошенническим.

Такие ошибки менее критичны, чем false negatives, но они могут создавать лишнюю ручную проверку.

In [34]:
false_positives[["text", "true_label", "pred_label"]].head(5)

,text,true_label,pred_label
6585,director of quality lightyear consulting llc i...,0,1
1658,data entry prepares source data for computer e...,0,1
17869,sr technical lead lims job title sr technical ...,0,1
3673,supply chain help desk this role will be in bu...,0,1
4656,call center manager complex care solutions url...,0,1


False negatives — это мошеннические вакансии, которые модель не смогла обнаружить.

Для fraud detection это наиболее критичный тип ошибок, так как такие вакансии остаются незамеченными.

In [35]:
false_negatives[["text", "true_label", "pred_label"]].head(5)

,text,true_label,pred_label
17648,vemma brand partner looking for motivated and ...,1,0
12104,production manager heavy duty diesel 2022 2022...,1,0
17549,sales manager collaborates with insert title i...,1,0
17598,real estate insurance professionals elite real...,1,0
6974,manager of project management organization eng...,1,0


Финальная модель допустила небольшое количество ошибок: `8` false positives и `18` false negatives.

False positives означают, что некоторые реальные вакансии были ошибочно помечены как мошеннические.  
False negatives более критичны для задачи fraud detection, так как это мошеннические вакансии, которые модель пропустила.

В дальнейшем качество модели можно улучшать за счёт:
- добавления структурных признаков вакансии;
- настройки порога классификации;
- более сложной обработки текста;
- использования ансамблей или transformer-based моделей.

Эксперимент со stop words

Проверим, улучшит ли качество удаление английских стоп-слов при TF-IDF-векторизации.

Стоп-слова — это частые слова вроде `the`, `and`, `of`, которые часто встречаются в текстах, но обычно несут мало информации для классификации.

Для эксперимента используем ту же модель `Linear SVC` и лучшие параметры, найденные ранее, но добавим `stop_words="english"` в `TfidfVectorizer`.

In [36]:
svc_stopwords_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=1,
        stop_words="english",
    )),
    ("clf", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

svc_stopwords_result = evaluate_model(
    "Linear SVC tuned + stopwords",
    svc_stopwords_pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
)

svc_stopwords_result

{'model': 'Linear SVC tuned + stopwords',
 'accuracy': 0.9916107382550335,
 'precision_fraud': 0.949685534591195,
 'recall_fraud': 0.8728323699421965,
 'f1_fraud': 0.9096385542168675,
 'roc_auc': 0.9917464868638519}

ывод по эксперименту со stop words

Удаление английских stop words не улучшило качество модели.

Модель `Linear SVC tuned + stopwords` показала более низкий `F1-score` для fraud-класса: `0.910` против `0.923` у финальной модели без удаления stop words.

Также снизился recall fraud-класса: `0.873` против `0.896`.

Поэтому в финальном решении stop words не удаляются. Вероятно, часть частотных слов всё же помогает модели различать стиль описания реальных и мошеннических вакансий.

In [37]:
final_results = pd.concat([
    final_results,
    pd.DataFrame([svc_stopwords_result])
], ignore_index=True)

final_results = final_results.sort_values(by="f1_fraud", ascending=False)
final_results

,model,accuracy,precision_fraud,recall_fraud,f1_fraud,roc_auc
0,Linear SVC tuned,0.992729,0.950920,0.895954,0.922619,0.991971
7,Linear SVC tuned + stopwords,0.991611,0.949686,0.872832,0.909639,0.991746
1,Logistic Regression tuned,0.990492,0.892655,0.913295,0.902857,0.992135
2,Linear SVC,0.990492,0.931677,0.867052,0.898204,0.991913
3,Logistic Regression,0.982662,0.768116,0.919075,0.836842,0.990486
4,SGDClassifier,0.977908,0.707965,0.924855,0.802005,0.989995
5,Complement NB,0.964206,0.680000,0.491329,0.570470,0.943790
6,Multinomial NB,0.962248,0.827586,0.277457,0.415584,0.943790


Эксперимент с character n-grams

Дополнительно проверим TF-IDF по символьным n-граммам.

В отличие от word-level TF-IDF, character n-grams анализируют не целые слова, а последовательности символов.  
Это может быть полезно для fraud detection, так как такой подход способен улавливать:
- необычные шаблоны написания;
- опечатки;
- части email/URL;
- подозрительные формулировки и повторяющиеся паттерны.

In [38]:
svc_char_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        max_features=50000,
        min_df=2,
    )),
    ("clf", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

svc_char_result = evaluate_model(
    "Linear SVC char ngrams",
    svc_char_pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
)

svc_char_result

{'model': 'Linear SVC char ngrams',
 'accuracy': 0.9862975391498882,
 'precision_fraud': 0.8690476190476191,
 'recall_fraud': 0.8439306358381503,
 'f1_fraud': 0.8563049853372433,
 'roc_auc': 0.987141573484124}

Вывод по эксперименту с character n-grams

Эксперимент с символьными n-граммами не улучшил качество модели.

`Linear SVC char ngrams` показала:
- `F1 fraud = 0.856`;
- `Precision fraud = 0.869`;
- `Recall fraud = 0.844`;
- `ROC-AUC = 0.987`.

Результат хуже, чем у финальной word-level TF-IDF модели.  
Вероятно, для данного датасета ключевая информация лучше выражена на уровне слов и словосочетаний, а не на уровне символов.

Поэтому в финальном решении используется word-level TF-IDF с униграммами и биграммами.

In [39]:
final_results = pd.concat([
    final_results,
    pd.DataFrame([svc_char_result])
], ignore_index=True)

final_results = final_results.sort_values(by="f1_fraud", ascending=False)
final_results

,model,accuracy,precision_fraud,recall_fraud,f1_fraud,roc_auc
0,Linear SVC tuned,0.992729,0.950920,0.895954,0.922619,0.991971
1,Linear SVC tuned + stopwords,0.991611,0.949686,0.872832,0.909639,0.991746
2,Logistic Regression tuned,0.990492,0.892655,0.913295,0.902857,0.992135
3,Linear SVC,0.990492,0.931677,0.867052,0.898204,0.991913
8,Linear SVC char ngrams,0.986298,0.869048,0.843931,0.856305,0.987142
4,Logistic Regression,0.982662,0.768116,0.919075,0.836842,0.990486
5,SGDClassifier,0.977908,0.707965,0.924855,0.802005,0.989995
6,Complement NB,0.964206,0.680000,0.491329,0.570470,0.943790
7,Multinomial NB,0.962248,0.827586,0.277457,0.415584,0.943790


Сохранение обновлённой таблицы экспериментов

После дополнительных NLP-экспериментов со stop words и character n-grams обновим итоговую таблицу экспериментов.

Финальной моделью остаётся `Linear SVC tuned`, так как дополнительные эксперименты не улучшили `F1-score` для fraud-класса.

In [40]:
final_results.to_csv(REPORTS_DIR / "experiments.csv", index=False)

print("Updated experiments saved to:", REPORTS_DIR / "experiments.csv")

Updated experiments saved to: ..\report\experiments.csv


Итоговый вывод

В рамках CP2 были проведены расширенные NLP-эксперименты для задачи классификации мошеннических вакансий.

Были протестированы несколько моделей машинного обучения на TF-IDF-признаках:
- Logistic Regression;
- Linear SVC;
- Multinomial Naive Bayes;
- Complement Naive Bayes;
- SGDClassifier.

Лучшей моделью стала `Linear SVC` после подбора гиперпараметров.

Финальная конфигурация:
- `TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=1)`;
- `LinearSVC(C=1.0, class_weight="balanced")`.

Качество финальной модели на тестовой выборке:
- `accuracy = 0.993`;
- `precision fraud = 0.951`;
- `recall fraud = 0.896`;
- `F1 fraud = 0.923`;
- `ROC-AUC = 0.992`.

По сравнению с CP1 качество модели заметно улучшилось.  
Дополнительные эксперименты со stop words и character n-grams не улучшили результат, поэтому в финальном решении используется word-level TF-IDF с униграммами и биграммами.

Финальная модель сохранена в `models/final_tfidf_linear_svc.joblib`, а таблица экспериментов сохранена в `report/experiments.csv`.